# 📚 Notebook 3 : Léa lit la documentation et cherche sur le web

**Formation « Comprendre et développer des agents IA spécialisés »**, module 2.

## L'histoire

Léa sait converser (notebook 1) et consulter le stock (notebook 2). Mais les clients posent aussi des questions **dont la réponse est dans les documents** de MégaShop :

- *« Mes AirPods achetés il y a 14 mois ne tiennent plus la charge : c'est couvert ? »* (conditions de garantie)
- *« Je fais du M, quelle taille de hoodie prendre ? »* (guide des tailles)
- *« Mes MégaPoints expirent quand ? »* (programme fidélité)

Et parfois des questions **dont la réponse est sur Internet** : *« Les AirPods sont-ils moins chers ailleurs ? »*

Tu vas apprendre à Léa à **chercher dans la documentation** (c'est le **RAG**) et **sur le web**, et à **choisir la bonne source**.

## 🎯 Ce que tu vas apprendre

| Étape | Thème | Python pur | LangChain |
|---|---|:---:|:---:|
| 0 | Configuration | ✅ | |
| 1 | La documentation de MégaShop | ✅ | |
| 2 | Le problème : le LLM ne connaît pas tes règles | ✅ | |
| 3 | Les **embeddings** : transformer un texte en nombres | ✅ | |
| 4 | Découper et indexer les documents | ✅ | |
| 5 | Rechercher, puis répondre : le **RAG** | ✅ | ✅ |
| 6 | Une réponse structurée avec **Pydantic** | ✅ | ✅ |
| 7 | Chercher sur le **web** | ✅ | |
| 8 | L'agent choisit sa source | ✅ | ✅ |
| 9 | Routeur à règles ou agent ? | ✅ | |

▶️ Exécute les cellules **dans l'ordre**, avec `Maj + Entrée`. Tout est dans ce notebook.

Dans les traces de l'agent, tu verras 4 symboles :

| Symbole | Signification |
|---|---|
| 🧠 | L'agent réfléchit |
| 🔧 | L'agent appelle un outil |
| 📦 | L'agent reçoit le résultat de l'outil |
| 💬 | L'agent répond au client |

---
## Étape 0/9 · Configuration  `▱▱▱▱▱▱▱▱▱`

🎯 **Objectif** : se connecter à OpenAI.

👉 Il faut une clé API dans un fichier `.env` à la racine du dossier, avec cette ligne : `OPENAI_API_KEY=sk-...`

On utilise **trois modèles**, chacun pour son rôle :

| Modèle | Rôle |
|---|---|
| `gpt-5-mini` | Raisonner et répondre au client |
| `text-embedding-3-small` | Transformer un texte en vecteur (étape 3) |
| `gpt-4.1-mini` + l'outil `web_search` | Chercher sur Internet, rapidement (étape 7) |

In [5]:
import os, json
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                   # lit la clé OPENAI_API_KEY dans le fichier .env
assert os.getenv("OPENAI_API_KEY"), "❌ Ajoute OPENAI_API_KEY=sk-... dans le fichier .env, puis redémarre le noyau"

client = OpenAI()
MODELE = "gpt-5-mini"
MODELE_EMBED = "text-embedding-3-small"
MODELE_WEB = "gpt-4.1-mini"                    # utilisé avec l'outil web_search d'OpenAI
print("✅ Connecté à OpenAI")

✅ Connecté à OpenAI


In [6]:
ICONES = {"pensee": "🧠 Réfléchit", "outil": "🔧 Appelle", "resultat": "📦 Reçoit", "reponse": "💬 Répond"}

def afficher(etape, texte, longueur_max=250):
    """Affiche une ligne de trace. Les résultats d'outils trop longs sont raccourcis."""
    texte = str(texte)
    if etape == "resultat" and len(texte) > longueur_max:
        texte = texte[:longueur_max].replace("\n", " ") + " […]"
    if etape == "reponse":
        print()
    print(f"{ICONES[etape]:<12} │ {texte}")

---
## Étape 1/9 · La documentation de MégaShop  `▰▱▱▱▱▱▱▱▱`

🎯 **Objectif** : préparer les documents que Léa devra consulter.

Dans une vraie entreprise, ce sont des pages web, des PDF ou un wiki. Ici, ce sont 5 petits documents en **Markdown**, rangés dans un dictionnaire. Chaque titre `##` marque une section.

💡 Ce sont des cellules de **données** : exécute-les sans tout lire.

In [7]:
DOCS = {}
DOCS["retours-remboursements"] = """
## Délai de retour
Vous disposez de 30 jours après réception pour retourner un article. Le retour est gratuit en point relais.
## Conditions de retour
L'article doit être non utilisé, complet et dans son emballage d'origine. Pour des raisons d'hygiène, les écouteurs
(dont les AirPods) ne sont repris que si leur emballage est encore scellé.
## Articles en promotion
Les articles achetés en promotion ou pendant les soldes se retournent dans les mêmes conditions, et sont remboursés au prix payé.
## Remboursement
Le remboursement est fait sous 5 jours ouvrés après réception du retour, sur le moyen de paiement utilisé.
En cas de paiement en 3 fois, les échéances restantes sont annulées.
"""
DOCS["garantie-sav"] = """
## Garantie légale
Tous les produits bénéficient de la garantie légale de conformité de 2 ans à partir de la livraison :
réparation ou remplacement gratuit en cas de panne non causée par le client.
## Batterie
Sur les smartphones, écouteurs et montres, la batterie est couverte pendant 2 ans si sa capacité tombe sous 80 %.
Au-delà de 2 ans, MégaShop propose un remplacement de batterie à prix fixe : 49 € pour les écouteurs, 89 € pour les smartphones.
## Extension MégaCare
L'extension MégaCare (29 €, à prendre lors de l'achat) prolonge la garantie à 3 ans et couvre une casse accidentelle.
## Faire une demande SAV
Depuis « Mes commandes », cliquez sur « Signaler un problème » et joignez une photo ou une vidéo. Réponse sous 48 h.
"""
DOCS["livraison"] = """
## Délais et tarifs
Livraison standard en 2 à 3 jours ouvrés : gratuite dès 100 €, sinon 4,90 €. Express en 24 h : 9,90 €, pour toute commande passée avant 14 h.
## Suivi du colis
Un lien de suivi est envoyé par e-mail dès l'expédition. Il est aussi visible dans « Mes commandes ».
## Colis endommagé
Refusez le colis ou signalez-le dans les 48 h avec des photos : MégaShop renvoie un article neuf sans frais.
## Livraison à l'étranger
Belgique, Luxembourg et Suisse : 4 à 6 jours ouvrés, 12,90 €. Pas de livraison hors Europe.
"""

In [8]:
DOCS["guide-tailles"] = """
## Baskets Nike Air Force 1
Elles taillent normalement : prenez votre pointure habituelle. Entre deux pointures, prenez la plus grande.
## Hoodie Adidas Essentials
Il taille grand : prenez une taille en dessous de votre taille habituelle pour une coupe ajustée,
ou votre taille habituelle pour un style oversize.
## Jean Levi's 501
Coupe droite. Correspondances : 36 = W27, 38 = W29, 40 = W31, 42 = W33, 44 = W35.
"""
DOCS["fidelite-megapoints"] = """
## Gagner des MégaPoints
1 € dépensé = 1 MégaPoint. Chaque avis client publié rapporte 20 points.
## Utiliser ses points
500 points = un bon d'achat de 10 €, utilisable dès 30 € d'achat. Les points expirent après 12 mois sans achat.
## Statut Méga+
Au-delà de 1 500 points cumulés dans l'année, vous passez Méga+ : livraison express offerte et accès aux ventes privées.
"""
print(f"📚 {len(DOCS)} documents :", ", ".join(DOCS))

📚 5 documents : retours-remboursements, garantie-sav, livraison, guide-tailles, fidelite-megapoints


---
## Étape 2/9 · Le problème : le LLM ne connaît pas tes règles  `▰▰▱▱▱▱▱▱▱`

🎯 **Objectif** : constater qu'un LLM seul répond « en général », pas selon les règles de MégaShop.

In [9]:
QUESTION_GARANTIE = "J'ai acheté des AirPods chez MégaShop il y a 14 mois et la batterie ne tient plus. C'est couvert ?"

r = client.chat.completions.create(model=MODELE, messages=[{"role": "user", "content": QUESTION_GARANTIE}])
afficher("reponse", r.choices[0].message.content)


💬 Répond     │ Probablement oui — mais ça dépend du pays et des garanties souscrites. Quelques précisions rapides et les étapes à suivre.

1) Où êtes‑vous (pays) ? Et avez‑vous le ticket/facture d’achat ?  
Ces informations déterminent exactement vos droits.

2) Si vous êtes en France (ou dans l’Union européenne)  
- Vous bénéficiez d’une garantie légale de conformité de 2 ans à partir de la livraison. Comme le problème est apparu après 14 mois, il entre en principe dans cette garantie.  
- Le vendeur (MégaShop) doit réparer ou remplacer le produit gratuitement. Si réparation/remplacement impossible, vous pouvez demander un remboursement partiel ou total.  
- Pour les défauts manifestes dans les 24 premiers mois, on suppose qu’ils existaient au moment de la livraison (c’est au vendeur de prouver le contraire).  
- Il existe aussi la garantie des vices cachés (action possible dans les 2 ans suivant la découverte), ce qui est une autre voie si nécessaire.  
- Si vous aviez pris AppleCar

🤔 La réponse est **générale** : le LLM parle de la garantie légale « en théorie », d'Apple, ou demande des précisions. Il ne peut pas savoir que chez MégaShop **la batterie est couverte 2 ans sous 80 % de capacité**.

💡 **L'idée du RAG** (*Retrieval-Augmented Generation*, c'est-à-dire génération augmentée par la recherche) : avant de répondre, on **cherche les passages utiles** dans la documentation, et on les **donne au LLM** avec la question.

```
 Question ──▶ 🔍 chercher dans les docs ──▶ 3 passages utiles ──▶ LLM ──▶ réponse fondée + sources
```

Mais comment trouver les « passages utiles » ? Une recherche par mot-clé ne suffit pas : le client écrit « batterie ne tient plus », le document dit « capacité tombe sous 80 % ». Il faut chercher **par le sens**.

---
## Étape 3/9 · Les embeddings : transformer un texte en nombres  `▰▰▰▱▱▱▱▱▱`

🎯 **Objectif** : comprendre comment un ordinateur compare le **sens** de deux phrases.

Un **embedding** transforme un texte en une liste de nombres (un **vecteur**). Deux textes au **sens proche** donnent des vecteurs **proches**, même s'ils n'ont aucun mot en commun.

In [10]:
def vectoriser(textes):
    """Transforme une liste de textes en vecteurs : une ligne de nombres par texte."""
    r = client.embeddings.create(model=MODELE_EMBED, input=textes)
    return np.array([d.embedding for d in r.data])

phrases = ["Je veux renvoyer mon colis",
           "Comment me faire rembourser ?",
           "Ces baskets taillent grand",
           "Quel temps fera-t-il demain ?"]
vecteurs = vectoriser(phrases)
print(f"{len(phrases)} phrases → {vecteurs.shape[1]} nombres par phrase")
print("Début du premier vecteur :", vecteurs[0][:6].round(3))

4 phrases → 1536 nombres par phrase
Début du premier vecteur : [ 0.031  0.04  -0.04  -0.061  0.003 -0.01 ]


Comparons maintenant chaque phrase à toutes les autres. Le score de **similarité** va de 0 (aucun rapport) à 1 (même sens) :

In [11]:
similarites = vecteurs @ vecteurs.T         # produit scalaire = similarité (ces vecteurs sont déjà normalisés)

print(" " * 34 + "".join(f"P{i + 1:<6}" for i in range(len(phrases))))
for i, phrase in enumerate(phrases):
    print(f"P{i + 1} {phrase:<31}" + "".join(f"{s:<7.2f}" for s in similarites[i]))

                                  P1     P2     P3     P4     
P1 Je veux renvoyer mon colis     1.00   0.51   0.29   0.15   
P2 Comment me faire rembourser ?  0.51   1.00   0.21   0.20   
P3 Ces baskets taillent grand     0.29   0.21   1.00   0.12   
P4 Quel temps fera-t-il demain ?  0.15   0.20   0.12   1.00   


👀 « Renvoyer mon colis » et « me faire rembourser » sont **proches**, alors qu'ils n'ont aucun mot en commun. La météo n'a rien à voir avec le reste.

✅ **À retenir** : un embedding capture le **sens** d'un texte. Comparer deux vecteurs, c'est comparer deux sens.

---
## Étape 4/9 · Découper et indexer les documents  `▰▰▰▰▱▱▱▱▱`

🎯 **Objectif** : préparer la documentation pour la recherche.

1. **Découper** : un document entier est trop long et parle de trop de choses. On le coupe en **passages** courts, ici une section `##` par passage.
2. **Indexer** : on calcule **une fois pour toutes** l'embedding de chaque passage.

In [12]:
PASSAGES = []
for nom_doc, texte in DOCS.items():
    for section in texte.split("## ")[1:]:                  # chaque section « ## Titre » devient un passage
        titre, _, contenu = section.partition("\n")
        PASSAGES.append({"source": f"{nom_doc} › {titre.strip()}",
                         "texte": f"{titre.strip()} : {contenu.strip()}"})

print(f"✂️ {len(PASSAGES)} passages. Exemple :")
print(PASSAGES[5])

✂️ 18 passages. Exemple :
{'source': 'garantie-sav › Batterie', 'texte': 'Batterie : Sur les smartphones, écouteurs et montres, la batterie est couverte pendant 2 ans si sa capacité tombe sous 80 %.\nAu-delà de 2 ans, MégaShop propose un remplacement de batterie à prix fixe : 49 € pour les écouteurs, 89 € pour les smartphones.'}


In [13]:
INDEX = vectoriser([p["texte"] for p in PASSAGES])         # un seul appel pour tous les passages
print(f"🗂️ Index prêt : {INDEX.shape[0]} passages × {INDEX.shape[1]} nombres")

🗂️ Index prêt : 18 passages × 1536 nombres


✅ **À retenir** : on découpe les documents en passages, puis on calcule leurs embeddings **une seule fois**. En production, on les range dans une **base vectorielle** (Chroma, pgvector, Pinecone...).

---
## Étape 5/9 · Rechercher, puis répondre : le RAG  `▰▰▰▰▰▱▱▱▱`

🎯 **Objectif** : retrouver les bons passages pour une question, puis répondre avec.

### 🔍 La recherche
On vectorise la question, puis on garde les **k passages les plus proches**.

In [14]:
def rechercher(question, k=3):
    """Renvoie les k passages dont le sens est le plus proche de la question, avec leur score."""
    scores = INDEX @ vectoriser([question])[0]
    meilleurs = np.argsort(scores)[::-1][:k]                # les indices des k meilleurs scores
    return [(round(float(scores[i]), 2), PASSAGES[i]) for i in meilleurs]

for score, passage in rechercher("Mes AirPods ne tiennent plus la charge"):
    print(f"{score}  📄 {passage['source']}")

0.49  📄 retours-remboursements › Conditions de retour
0.45  📄 garantie-sav › Batterie
0.34  📄 guide-tailles › Baskets Nike Air Force 1


👀 Le passage **« Batterie »** est retrouvé, alors que la question ne contient ni « batterie » ni « capacité ». C'est la recherche **par le sens**.
Le passage sur les **conditions de retour** remonte aussi, car il parle d'AirPods. C'est pour ça qu'on garde **plusieurs passages** (k = 3) : le LLM fera le tri.

### 💬 La réponse
On donne les passages au LLM, avec une consigne stricte : **répondre uniquement à partir des extraits** et **citer ses sources**.

In [15]:
SYSTEME_RAG = """Tu es Léa, conseillère de MégaShop. Réponds UNIQUEMENT à partir des extraits fournis.
Cite tes sources entre crochets, par exemple [garantie-sav › Batterie].
Si les extraits ne permettent pas de répondre, dis-le et propose de transférer à un conseiller."""

def repondre_avec_docs(question):
    extraits = "\n\n".join(f"[{p['source']}]\n{p['texte']}" for _, p in rechercher(question))
    r = client.chat.completions.create(model=MODELE, messages=[
        {"role": "system", "content": SYSTEME_RAG},
        {"role": "user", "content": f"Extraits :\n{extraits}\n\nQuestion du client : {question}"}])
    return r.choices[0].message.content

afficher("reponse", repondre_avec_docs(QUESTION_GARANTIE))


💬 Répond     │ Bonjour — je suis Léa. Oui : les écouteurs (AirPods) sont couverts pendant 2 ans si la capacité de la batterie tombe sous 80 %. Vous êtes à 14 mois, donc si la capacité est < 80 % la prise en charge s'applique [garantie-sav › Batterie].  

Si vous envisagez un retour plutôt qu'une prise en charge SAV, notez que pour des raisons d'hygiène les écouteurs ne sont repris que si leur emballage est encore scellé [retours-remboursements › Conditions de retour].


Et si la réponse **n'est pas** dans la documentation ?

In [16]:
afficher("reponse", repondre_avec_docs("Est-ce que je peux payer en cryptomonnaie ?"))


💬 Répond     │ Bonjour — d’après les extraits fournis ([retours-remboursements › Remboursement], [retours-remboursements › Articles en promotion], [fidelite-megapoints › Utiliser ses points]), aucun ne traite des moyens de paiement ni de la cryptomonnaie. Je ne peux donc pas répondre à votre question avec ces documents.

Souhaitez‑vous que je transfère votre demande à un conseiller pour vérification ? — Léa


👍 Léa **reconnaît qu'elle ne sait pas** au lieu d'inventer. C'est le comportement attendu, grâce à la consigne « UNIQUEMENT à partir des extraits ».

### 🔁 La recherche avec LangChain

LangChain fournit des **bases vectorielles** prêtes à l'emploi. `InMemoryVectorStore` fait exactement ce que font nos `INDEX` et `rechercher()`. Pour passer à une vraie base comme Chroma ou pgvector, on change une seule ligne.

In [17]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

base = InMemoryVectorStore.from_texts([p["texte"] for p in PASSAGES], OpenAIEmbeddings(model=MODELE_EMBED),
                                      metadatas=[{"source": p["source"]} for p in PASSAGES])

for doc in base.similarity_search("Mes AirPods ne tiennent plus la charge", k=3):
    print("📄", doc.metadata["source"])

📄 retours-remboursements › Conditions de retour
📄 garantie-sav › Batterie
📄 guide-tailles › Baskets Nike Air Force 1


✅ **À retenir** : le RAG, c'est **chercher** les passages utiles, puis les **injecter** dans le prompt. Le LLM répond alors selon **tes** documents, et cite ses sources.

---
## Étape 6/9 · Une réponse structurée avec Pydantic  `▰▰▰▰▰▰▱▱▱`

🎯 **Objectif** : que la réponse soit **exploitable par le code**, et pas seulement lisible par le client.

Une réponse en texte libre, c'est bien pour le client. Mais le site de MégaShop a besoin de savoir : **quelles sources** afficher ? La réponse a-t-elle été **trouvée** ? Faut-il **transférer à un humain** ?

On décrit ce format avec **Pydantic**, et on oblige le LLM à le respecter :

In [18]:
from pydantic import BaseModel, Field

class ReponseSAV(BaseModel):
    reponse: str = Field(description="Réponse au client, en 3 phrases maximum")
    sources: list[str] = Field(description="Sources utilisées, par exemple 'garantie-sav › Batterie'")
    trouve_dans_docs: bool = Field(description="False si les extraits ne permettent pas de répondre")
    transfert_humain: bool = Field(description="True si un conseiller humain doit reprendre la demande")

In [19]:
def repondre_structure(question):
    extraits = "\n\n".join(f"[{p['source']}]\n{p['texte']}" for _, p in rechercher(question))
    r = client.chat.completions.parse(model=MODELE, response_format=ReponseSAV, messages=[
        {"role": "system", "content": SYSTEME_RAG},
        {"role": "user", "content": f"Extraits :\n{extraits}\n\nQuestion du client : {question}"}])
    return r.choices[0].message.parsed                    # un objet ReponseSAV validé par Pydantic

for question in [QUESTION_GARANTIE, "Est-ce que je peux payer en cryptomonnaie ?"]:
    rep = repondre_structure(question)
    print(f"👤 {question}\n💬 {rep.reponse}\n📄 Sources : {rep.sources}")
    print("📞 → Ticket créé pour un conseiller\n" if rep.transfert_humain else "✅ → Réponse envoyée au client\n")

👤 J'ai acheté des AirPods chez MégaShop il y a 14 mois et la batterie ne tient plus. C'est couvert ?
💬 Oui — les AirPods sont couverts pendant 2 ans si la capacité de la batterie est tombée sous 80 % ; à 14 mois, cela entre dans la période de couverture si la capacité est inférieure à 80 %.
📄 Sources : ['garantie-sav › Batterie']
✅ → Réponse envoyée au client

👤 Est-ce que je peux payer en cryptomonnaie ?
💬 Désolée, les extraits fournis ([retours-remboursements › Remboursement], [retours-remboursements › Articles en promotion], [fidelite-megapoints › Utiliser ses points]) ne mentionnent pas le paiement en cryptomonnaie. Je peux transférer votre demande à un conseiller pour confirmation. Voulez-vous que je le fasse ?
📄 Sources : ['retours-remboursements › Remboursement', 'retours-remboursements › Articles en promotion', 'fidelite-megapoints › Utiliser ses points']
📞 → Ticket créé pour un conseiller



👀 Le **code** décide de la suite (envoyer la réponse ou créer un ticket) grâce aux champs `transfert_humain` et `sources`, sans avoir à « deviner » en lisant le texte.

### 🔁 La même chose avec LangChain

In [20]:
llm = ChatOpenAI(model=MODELE)
chaine_rag = llm.with_structured_output(ReponseSAV)

question = "Je fais du M d'habitude, quelle taille prendre pour le hoodie Adidas ?"
extraits = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in base.similarity_search(question, k=3))
rep = chaine_rag.invoke([("system", SYSTEME_RAG), ("user", f"Extraits :\n{extraits}\n\nQuestion du client : {question}")])
print(f"💬 {rep.reponse}\n📄 {rep.sources}")

💬 Si vous faites du M d'habitude, prenez une taille en dessous (S) pour une coupe ajustée. Si vous préférez un style oversize, prenez votre taille habituelle (M).
📄 ['guide-tailles \x187 Hoodie Adidas Essentials\x187']


✅ **À retenir** : avec Pydantic, la réponse du LLM devient une **donnée fiable** que ton application peut utiliser (afficher les sources, router vers un humain...).

---
## Étape 7/9 · Chercher sur le web  `▰▰▰▰▰▰▰▱▱`

🎯 **Objectif** : répondre aux questions dont la réponse **n'est pas** chez MégaShop.

*« Les AirPods sont-ils moins chers ailleurs ? »* : aucun document interne ne le dira. Il faut **Internet**.

OpenAI propose un outil **`web_search`** : le modèle cherche lui-même sur Internet et cite ses sources. Il s'utilise avec l'API **Responses**, une autre interface d'OpenAI que `chat.completions`. On l'enveloppe dans une fonction, qui deviendra un **outil** pour Léa.

In [21]:
def recherche_web(requete: str) -> str:
    """Cherche sur Internet des informations publiques et récentes : prix chez les concurrents, tests de produits, actualités."""
    r = client.responses.create(
        model=MODELE_WEB, tools=[{"type": "web_search"}],     # 👈 active la recherche web
        input=f"{requete}\nRéponds en 5 lignes maximum, en français, et cite tes sources.")
    return r.output_text

print(recherche_web("Prix des AirPods 4 chez les grands revendeurs en France"))

Les AirPods 4 sont disponibles chez plusieurs grands revendeurs en France. Par exemple, Carrefour propose le modèle standard à 128,99 € ([dodeal.fr](https://dodeal.fr/product/airpods-4-carrefour-0195949688584?utm_source=openai)). La Fnac offre le modèle standard à 96,90 € ([comparr.fr](https://www.comparr.fr/products/apple-airpods-4?utm_source=openai)). Cdiscount vend le modèle avec réduction active du bruit à 174 € ([prixairpods.fr](https://prixairpods.fr/produit/airpods-4-anc?utm_source=openai)). Reepeat propose des AirPods 4 reconditionnés à partir de 61,96 € ([reepeat.fr](https://www.reepeat.fr/products/p/apple-airpods-4?utm_source=openai)). Les prix peuvent varier en fonction de l'état du produit et des promotions en cours. 


⚠️ **Le web demande de la prudence**
- 🌍 C'est une **vraie** recherche : le résultat change d'un jour à l'autre.
- 🤷 Une page web peut être **fausse ou périmée**. Il faut toujours **citer la source**.
- 🏠 Pour les **règles de MégaShop** (retours, garantie...), la documentation interne reste la **seule source de vérité**. Un site externe ne doit jamais la contredire.
- 🪤 Une page web peut contenir des **instructions piégées** destinées à l'IA. C'est l'**injection de prompt**, qu'on verra au notebook 4.

---
## Étape 8/9 · L'agent choisit sa source  `▰▰▰▰▰▰▰▰▱`

🎯 **Objectif** : donner à Léa **deux outils** (documentation et web), et la laisser **choisir** selon la question.

On reprend la **boucle ReAct** du notebook 2. Pour l'agent, le RAG n'est qu'**un outil de plus**.

In [22]:
def rechercher_docs(question: str) -> str:
    """Cherche dans la documentation de MégaShop : retours, garantie, SAV, livraison, guide des tailles, fidélité."""
    return "\n\n".join(f"[{p['source']}] {p['texte']}" for _, p in rechercher(question))

OUTILS = {"rechercher_docs": rechercher_docs, "recherche_web": recherche_web}
SCHEMAS = [{"type": "function", "function": {"name": nom, "description": f.__doc__, "parameters": {
               "type": "object", "properties": {p: {"type": "string"} for p in f.__annotations__ if p != "return"},
               "required": [p for p in f.__annotations__ if p != "return"]}}}
           for nom, f in OUTILS.items()]                  # même principe que decrire() du notebook 2

SYSTEME_AGENT = """Tu es Léa, conseillère de vente de MégaShop, une boutique 100 % en ligne. Tu as deux sources :
- rechercher_docs : les règles de MégaShop (retours, garantie, livraison, tailles, fidélité). C'est la source de vérité.
- recherche_web : Internet, pour les infos publiques (prix chez les concurrents, tests, actualités). Jamais pour les règles de MégaShop.
Si la question a plusieurs parties, utilise la bonne source pour chaque partie.
Cite tes sources. Réponds en français, en 5 phrases maximum, avec un ton chaleureux."""

### En Python pur : la boucle ReAct (reprise du notebook 2)

In [23]:
def agent_python(question, max_tours=6):
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    for _ in range(max_tours):
        msg = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:                                   # plus d'outil demandé -> réponse finale
            return afficher("reponse", msg.content)
        for appel in msg.tool_calls:
            arguments = json.loads(appel.function.arguments)
            afficher("outil", f"{appel.function.name}({arguments})")                # 🔧 AGIR
            resultat = OUTILS[appel.function.name](**arguments)
            afficher("resultat", resultat)                                          # 📦 OBSERVER
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})

In [24]:
print("════ Question sur une règle de MégaShop ════")
agent_python("Je fais du M d'habitude, quelle taille prendre pour le hoodie Adidas ?")

════ Question sur une règle de MégaShop ════
🔧 Appelle    │ rechercher_docs({'question': 'Si un client fait habituellement du M, quelle taille doit-il prendre pour le hoodie Adidas ? (guide des tailles MégaShop)'})
📦 Reçoit     │ [guide-tailles › Hoodie Adidas Essentials] Hoodie Adidas Essentials : Il taille grand : prenez une taille en dessous de votre taille habituelle pour une coupe ajustée, ou votre taille habituelle pour un style oversize.  [guide-tailles › Baskets Nike  […]

💬 Répond     │ Si vous faites habituellement du M pour le hoodie Adidas, prenez une taille en dessous (S) pour une coupe ajustée.  
Si vous préférez un aspect oversize, conservez le M.  
Source : guide-tailles › Hoodie Adidas Essentials (MégaShop).  
— Léa, conseillère MégaShop


In [25]:
print("════ La question fil rouge : deux sources ════")
FIL_ROUGE = ("Mes AirPods achetés chez vous il y a 14 mois ne tiennent plus la charge : c'est couvert ? "
             "Et si je dois en racheter, ils sont moins chers ailleurs que chez vous (149 €) ?")
agent_python(FIL_ROUGE)

════ La question fil rouge : deux sources ════
🔧 Appelle    │ rechercher_docs({'question': 'Garantie et SAV MégaShop pour écouteurs (AirPods) achetés il y a 14 mois qui ne tiennent plus la charge : la prise en charge est-elle possible ? Durée de garantie, conditions (défaut, usure de la batterie), procédures (SAV, échange, remboursement), et politique sur la garantie légale de conformité vs garantie commerciale.'})
📦 Reçoit     │ [garantie-sav › Batterie] Batterie : Sur les smartphones, écouteurs et montres, la batterie est couverte pendant 2 ans si sa capacité tombe sous 80 %. Au-delà de 2 ans, MégaShop propose un remplacement de batterie à prix fixe : 49 € pour les écouteur […]
🔧 Appelle    │ recherche_web({'requete': "AirPods prix 149 € vendeur concurrent '149 €' prix AirPods 2026 site marchand France"})
📦 Reçoit     │ Les AirPods 5 sont proposés à 149 € sur le site officiel d'Apple France. ([apple.com](https://www.apple.com/airpods/?utm_source=openai)) Ce modèle offre une annulatio

👀 Léa a utilisé **les deux outils** : la documentation pour la garantie, et le web pour les prix.

### 🔁 La même chose avec LangChain : `create_agent`

Au notebook 2, on a construit le graphe ReAct à la main avec LangGraph. `create_agent` construit **ce même graphe en une ligne**.

In [26]:
from langchain.agents import create_agent
from langchain_core.tools import tool

agent_lc = create_agent(model=llm, tools=[tool(rechercher_docs), tool(recherche_web)], system_prompt=SYSTEME_AGENT)

def afficher_messages(messages):
    """Traduit l'historique LangChain en traces 🔧 📦 💬."""
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            for appel in m.tool_calls:
                afficher("outil", f"{appel['name']}({appel['args']})")
        elif m.type == "tool":
            afficher("resultat", m.content)
        elif m.type == "ai":
            afficher("reponse", m.content)

In [27]:
afficher_messages(agent_lc.invoke({"messages": [("user", FIL_ROUGE)]})["messages"])

🔧 Appelle    │ rechercher_docs({'question': 'garantie batterie AirPods 14 mois couverture réparation remplacement SAV MégaShop politique retours garantie légale de conformité garantie commerciale délai 14 mois'})
🔧 Appelle    │ recherche_web({'requete': "AirPods price France 2026 prix 149 € competitors Fnac Darty Amazon Apple Store 'AirPods' prix"})
📦 Reçoit     │ [garantie-sav › Batterie] Batterie : Sur les smartphones, écouteurs et montres, la batterie est couverte pendant 2 ans si sa capacité tombe sous 80 %. Au-delà de 2 ans, MégaShop propose un remplacement de batterie à prix fixe : 49 € pour les écouteur […]
📦 Reçoit     │ En 2026, les AirPods 4 sont proposés à 149,99 € chez Darty ([darty.com](https://www.darty.com/nav/achat/accessoires/casque_ecouteurs/casque_intra-auriculaire/apple_airpods_4.html?msockid=08323da7df546b7c1f092b39ded16ab9&utm_source=openai)) et à 149,9 […]

💬 Répond     │ Bonjour — bonne nouvelle : vos AirPods achetés chez MégaShop il y a 14 mois sont couverts pa

✅ **À retenir**
- Pour un agent, le RAG est **un outil comme un autre**.
- La **description des outils** et le **prompt système** guident le choix de la source : *docs = règles de MégaShop, web = infos publiques*.

---
## Étape 9/9 · Routeur à règles ou agent ?  `▰▰▰▰▰▰▰▰▰`

🎯 **Objectif** : savoir quand **ne pas** laisser le choix au LLM.

Choisir la source peut aussi se faire **sans LLM**, avec de simples règles :

In [28]:
def routeur(question):
    """Choisit la source avec des mots-clés : rapide, gratuit et prévisible."""
    mots_web = ["ailleurs", "concurrent", "moins cher", "actualité", "test", "avis presse"]
    return "🌍 web" if any(mot in question.lower() for mot in mots_web) else "📚 docs"

for q in ["Quel est le délai de retour ?", "L'iPhone 16 est moins cher ailleurs ?",
          "Mes AirPods ne marchent plus, et ils sont moins chers ailleurs ?"]:
    print(f"{routeur(q):<8} ← {q}")

📚 docs   ← Quel est le délai de retour ?
🌍 web    ← L'iPhone 16 est moins cher ailleurs ?
🌍 web    ← Mes AirPods ne marchent plus, et ils sont moins chers ailleurs ?


👀 La 3ᵉ question a **besoin des deux sources**, mais le routeur n'en choisit qu'une. C'est la limite des règles fixes.

| | 🔀 Routeur à règles | 🤖 Agent |
|---|---|---|
| Coût | Gratuit | Un appel LLM par décision |
| Vitesse | Instantané | Quelques secondes |
| Prévisible | ✅ Toujours la même décision | ❌ Peut varier |
| Questions complexes | ❌ Une seule source | ✅ Combine plusieurs sources |

💡 **En pratique** : commence par le plus simple qui fonctionne. Les règles conviennent quand les cas sont connus et peu nombreux. L'agent se justifie quand les questions sont variées ou combinées.

---
## 🎓 Récapitulatif

| Concept | En une phrase | Python pur | LangChain |
|---|---|---|---|
| **Embedding** | Un texte transformé en vecteur qui capture son sens | `client.embeddings.create()` | `OpenAIEmbeddings` |
| **Index** | Les vecteurs de tous les passages, calculés une fois | `np.array` | `InMemoryVectorStore` |
| **RAG** | Chercher les passages utiles, puis les donner au LLM | `rechercher()` + prompt | `similarity_search()` |
| **Réponse structurée** | Une réponse exploitable par le code | `parse(response_format=...)` | `with_structured_output()` |
| **Agent multi-sources** | Le LLM choisit docs ou web | boucle ReAct | `create_agent` |

**Les 3 idées à garder**
1. Le RAG fait répondre le LLM **selon tes documents**, avec des sources.
2. On cherche **par le sens** (embeddings), pas par mots-clés.
3. Le web est utile mais **non fiable** : tes documents internes restent la source de vérité.

➡️ **Prochain notebook** : Léa devient **une équipe** (un agent Vente et un agent SAV), connectée aux systèmes de MégaShop par **MCP**, et on la **sécurise** contre les pièges.